# Classes first steps

This module is the first day with classes. It assumes the reader has
worked through `python_primer.md` and is comfortable with variables,
lists, dicts, and functions, but has never written a class before. The
focus is the smallest set of ideas that make a class useful: what
problem it solves, how to define one, and how to read the resulting
code. Inheritance, properties, dunder methods, and dataclasses are
deferred to `classes.md`, which assumes the material here.

A single Sales Plan example threads through every section. The class
under construction tracks one region's monthly actuals against a
target, so each topic adds one capability to the same `RegionPlan`
rather than introducing a new toy example.

The topics below are arranged linearly for review. Structural grouping
(sections, chapters) can be applied later.

---

## Topic list

1. What classes are for
2. Life before classes
3. Defining a class
4. `__init__` and instance attributes
5. What `self` actually is
6. Methods
7. Many instances at once
8. Printing instances with `__repr__`
9. Where this primer stops
10. Starter habits
11. Common mistakes

---

## 1. What classes are for

A class is a way to bundle related data with the operations that work
on it, and to stamp out as many independent copies of that bundle as
the program needs. In a planning context the bundle is something like
"one region's sales plan": a name, a target, the actuals recorded so
far, and a small set of things you do with them (record a month, total
the actuals, compare against target). Without a class, those pieces
sit as separate variables and the operations are free standing
functions; with a class, the pieces and the operations live together
under one name and one type.

Two practical benefits follow. First, related state stops drifting
apart: a `RegionPlan` always carries its own target and its own
actuals, and you cannot accidentally add Europe's number to Asia's
total by passing the wrong dict into the wrong function. Second, the
operations get a natural home. Rather than `record_month(plan, "Jan",
12000.0)`, the call reads `plan.record("Jan", 12000.0)`. The function
is the same; the syntax announces which object it belongs to.

Classes are not the right answer for everything. A short script that
reads a CSV, adds a column, and writes it back does not need any
classes at all. Reach for a class when the same group of values keeps
appearing in the same set of functions, and especially when more than
one independent copy of that group needs to exist at once.

## 2. Life before classes

Suppose the task is to track Europe's sales plan: a target of 120,000
and a few months of actuals. The most direct version uses plain
variables.

In [ ]:
europe_target = 120_000.0
europe_actuals: dict[str, float] = {}

europe_actuals["Jan"] = 9_500.0
europe_actuals["Feb"] = 11_200.0

europe_total = sum(europe_actuals.values())
europe_gap = europe_target - europe_total
print(europe_gap)            # 99300.0

This works. It stops working as soon as a second region appears,
because every variable now needs a region prefix and every line of
code has to be duplicated. The first cleanup is a dict.

In [ ]:
plan = {
    "name": "Europe",
    "target": 120_000.0,
    "actuals": {},
}

plan["actuals"]["Jan"] = 9_500.0
plan["actuals"]["Feb"] = 11_200.0

total = sum(plan["actuals"].values())
gap = plan["target"] - total

Better. A second region is now `plan_americas = {...}` rather than a
new set of variables. Two problems remain. First, the operations
(record a month, compute total, compute gap) are still scattered free
standing functions that take `plan` as the first argument and reach
into it by string keys. A typo in `plan["targt"]` is a silent bug.
Second, nothing stops a caller from putting a list into `target` or
forgetting `actuals` entirely; the dict has no shape.

A class fixes both: the shape is fixed in one place, and the
operations belong to it.

## 3. Defining a class

A class is defined with the `class` keyword. The body is an indented
block, like a function body, and contains the methods that belong to
the class. By convention, class names use `PascalCase`.

In [ ]:
class RegionPlan:
    pass

`pass` is a do nothing placeholder; the class is legal but empty. An
instance is created by calling the class as if it were a function:

In [ ]:
plan = RegionPlan()
type(plan)            # <class '__main__.RegionPlan'>

`plan` is now an object whose type is `RegionPlan`. It has no useful
attributes yet, but the machinery for adding them is already present.
Attributes can be set on it from outside:

In [ ]:
plan.name = "Europe"
plan.target = 120_000.0
plan.actuals = {}

This is closer to the dict version than to a real class: the shape is
not fixed, and creating a new instance correctly takes three separate
assignments. The next topic addresses both at once.

## 4. `__init__` and instance attributes

`__init__` is a method that runs automatically when an instance is
created. It receives the new instance as its first argument and any
other arguments passed to the class call. Its job is to set the
attributes that every instance should start with.

In [ ]:
class RegionPlan:
    def __init__(self, name: str, target: float) -> None:
        self.name = name
        self.target = target
        self.actuals: dict[str, float] = {}


plan = RegionPlan("Europe", 120_000.0)
plan.name             # 'Europe'
plan.target           # 120000.0
plan.actuals          # {}

The arguments to `RegionPlan(...)` are passed straight through to
`__init__` (skipping `self`, which Python supplies automatically).
After `__init__` returns, the instance is fully formed: it has a
`name`, a `target`, and an empty `actuals` dict. Every `RegionPlan`
created the same way has the same set of attributes, so the rest of
the code can rely on that shape.

The leading and trailing double underscores in `__init__` mark it as
one of Python's "special methods", a small set of names the language
treats specially. There are many others (covered in `classes.md`);
`__init__` is the only one a beginner needs.

The pitfall worth noting now: every instance attribute should be
created in `__init__`, not later. If `actuals` is only created the
first time `record` is called, then code that asks for `plan.actuals`
on a fresh plan will raise `AttributeError`. Initialising every
attribute up front makes the class's shape a single readable block.

## 5. What `self` actually is

`self` is the parameter name, by strong convention, for "the instance
this method was called on". When `plan.record("Jan", 9500.0)` runs,
Python passes `plan` as the first argument to `record`; inside the
method, that first argument is bound to `self`, and assignments like
`self.actuals[month] = value` modify `plan` directly.

`self` is not a Python keyword. It is just the first parameter, and
calling it something else is legal but confusing.

In [ ]:
class RegionPlan:
    def __init__(this, name, target):       # legal but jarring
        this.name = name
        this.target = target

Languages such as Java and C# hide the equivalent (`this`) and inject
it implicitly. Python chooses the opposite: the instance is an
ordinary parameter. The two consequences are that every method must
declare `self` as its first parameter, and every reference to an
instance attribute inside a method goes through it: `self.target`,
not `target`. A bare `target` inside a method refers to a local
variable, never to the attribute, and forgetting `self.` is one of
the most common beginner mistakes (see Topic 11).

The same instance can be reached two ways:

In [ ]:
plan = RegionPlan("Europe", 120_000.0)

plan.record("Jan", 9_500.0)            # the usual form
RegionPlan.record(plan, "Jan", 9_500.0) # the explicit form, identical effect

The second form is rarely written by hand, but seeing it once makes
the implicit first argument less mysterious.

## 6. Methods

A method is a function defined inside a class body. It looks exactly
like a free standing function except for the leading `self`
parameter. Methods are how operations attach to a class: instead of
`record_month(plan, month, value)`, the code reads `plan.record(month,
value)`.

In [ ]:
class RegionPlan:
    def __init__(self, name: str, target: float) -> None:
        self.name = name
        self.target = target
        self.actuals: dict[str, float] = {}

    def record(self, month: str, value: float) -> None:
        if value < 0:
            raise ValueError("Actuals cannot be negative")
        self.actuals[month] = value

    def total(self) -> float:
        return sum(self.actuals.values())

    def gap(self) -> float:
        return self.target - self.total()


plan = RegionPlan("Europe", 120_000.0)
plan.record("Jan", 9_500.0)
plan.record("Feb", 11_200.0)
plan.total()              # 20700.0
plan.gap()                # 99300.0

Three things are worth noticing. First, `record` validates its input
and raises a normal Python exception if the input is wrong; methods
are functions and the same error handling applies. Second, `gap`
calls `self.total()`. Methods can call each other through `self`
exactly the way external code does, and the same dispatch rules
apply. Third, the operations now live next to the data they operate
on. Reading the class top to bottom answers "what is a `RegionPlan`?"
in one place: a name, a target, an actuals dict, and four operations.

## 7. Many instances at once

The whole point of a class is that creating a second instance is
cheap. Each instance has its own attributes; none of them shares
state with the others unless the code explicitly arranges for it.

In [ ]:
europe = RegionPlan("Europe", 120_000.0)
americas = RegionPlan("Americas", 250_000.0)
asia = RegionPlan("Asia", 90_000.0)

europe.record("Jan", 9_500.0)
americas.record("Jan", 22_300.0)
asia.record("Jan", 7_100.0)

europe.actuals          # {'Jan': 9500.0}
americas.actuals        # {'Jan': 22300.0}
asia.actuals            # {'Jan': 7100.0}

Because instances are ordinary Python values, they can be put in a
list and iterated like anything else.

In [ ]:
plans = [
    RegionPlan("Europe", 120_000.0),
    RegionPlan("Americas", 250_000.0),
    RegionPlan("Asia", 90_000.0),
]

for plan in plans:
    plan.record("Jan", 10_000.0)

for plan in plans:
    print(plan.name, plan.gap())
# Europe 110000.0
# Americas 240000.0
# Asia 80000.0

Compared to the dict version of the same code, two changes have
happened. The shape of each plan is fixed, so the loop body cannot
silently break by misspelling a key. And the operations are written
once, on the class, instead of as free standing functions that have
to remember the dict layout.

## 8. Printing instances with `__repr__`

Printing an instance straight out of the box gives an unhelpful
result.

In [ ]:
plan = RegionPlan("Europe", 120_000.0)
print(plan)
# <__main__.RegionPlan object at 0x10a9b3e50>

The default representation is the class name and a memory address.
Defining a `__repr__` method changes that. Like `__init__`,
`__repr__` is one of Python's special methods; it is called whenever
a value needs to be turned into a string for inspection (the REPL,
`print`, error messages).

In [ ]:
class RegionPlan:
    def __init__(self, name: str, target: float) -> None:
        self.name = name
        self.target = target
        self.actuals: dict[str, float] = {}

    def __repr__(self) -> str:
        return f"RegionPlan(name={self.name!r}, target={self.target}, actuals={self.actuals})"


plan = RegionPlan("Europe", 120_000.0)
plan.record("Jan", 9_500.0)
print(plan)
# RegionPlan(name='Europe', target=120000.0, actuals={'Jan': 9500.0})

A useful rule of thumb: `__repr__` should produce a string that
identifies the object precisely enough for a developer to recognise
it in a log line or a stack trace. Defining it early is cheap and
pays off the first time something goes wrong.

`__repr__` is the only special method this primer covers. The full
list (equality, ordering, container behaviour, arithmetic) is in
`classes.md`.

## 9. Where this primer stops

A `RegionPlan` with `__init__`, three methods, and a `__repr__` is a
complete, useful class. It is also close to the simplest interesting
class one can write. The rest of Python's class system extends what
is here in three directions, and `classes.md` covers each in turn.

The first direction is sharing code between similar classes through
**inheritance**: a `WeightedRegionPlan` that behaves like a
`RegionPlan` but applies a weighting factor to the target. The second
is making attribute access run code (validation, derived values)
through **properties**. The third is a longer list of special methods
that hook into language syntax: `__eq__` for equality, `__len__` for
`len(plan)`, `__iter__` for `for month in plan`, and several others.
Each is one more way of making a class behave like a built in type.

For most planning code, the material in this primer is enough. Reach
for the deeper material when a real need appears, not in advance.

## 10. Starter habits

A short collection of habits that make early class code pleasant to
read and modify. Each is one paragraph; together they form the
"taste" of writing a small Python class well.

**Define every instance attribute in `__init__`.** Even attributes
that start as `None` or an empty container belong in `__init__` so
that the class's shape is one readable block. Adding attributes
later, only when first used, fragments the shape across the file and
turns simple `print(plan)` calls into `AttributeError`.

**Type hint `__init__` and the methods.** Annotations are not
enforced at runtime, but they document the intended shape and let an
editor or `mypy` flag mistakes. `def record(self, month: str, value:
float) -> None:` is more useful than `def record(self, month, value):`
the moment the code is read by someone else, including a future self.

**Keep the class small.** A first version with three or four methods
is normal. When the class grows past a screenful, look for a method
group that uses only some of the attributes; that group is usually a
second class waiting to be extracted. Keeping classes small is one of
the design principles `classes.md` revisits in detail.

**Use realistic names.** `RegionPlan`, `Forecast`, `Account`,
`Transaction` are useful class names because they describe a domain
concept. `Manager`, `Helper`, `Handler`, `Util` describe nothing and
tend to attract every loose method in the file.

**Reach for a class when state and operations belong together.** A
single function that runs once and returns a number does not need a
class. A bundle of related values that several functions read and
modify usually does, and the moment a second copy of that bundle has
to exist, the class is overdue.

## 11. Common mistakes

A short collection of errors specific to first time class authors,
each shown as a `# Wrong` and `# Correct` pair. Most appear once in
every reader's history.

**Forgetting `self` on attribute access.** Inside a method, the
attribute is `self.target`, not `target`. A bare name is a local
variable.

In [ ]:
# Wrong
class RegionPlan:
    def __init__(self, name: str, target: float) -> None:
        self.name = name
        self.target = target

    def gap(self) -> float:
        return target - sum(self.actuals.values())   # NameError: target


# Correct
class RegionPlan:
    def __init__(self, name: str, target: float) -> None:
        self.name = name
        self.target = target

    def gap(self) -> float:
        return self.target - sum(self.actuals.values())

The error is loud (a `NameError` or `AttributeError` at call time),
which is the easy case. The harder case is assignment without
`self.`, which silently creates a local variable and leaves the
attribute untouched.

**Forgetting `self` in the method signature.** Methods always declare
`self` as their first parameter. Without it, the implicit instance
argument lands in the wrong slot.

In [ ]:
# Wrong
class RegionPlan:
    def total():                  # missing self
        return sum(self.actuals.values())


plan.total()                      # TypeError: total() takes 0 positional arguments but 1 was given


# Correct
class RegionPlan:
    def total(self) -> float:
        return sum(self.actuals.values())

**Mutable default in `__init__`.** A mutable default value is created
once, when the function is defined, and reused on every call. The
result is that every instance shares the same list or dict.

In [ ]:
# Wrong
class RegionPlan:
    def __init__(self, name: str, target: float, actuals: dict = {}) -> None:
        self.name = name
        self.target = target
        self.actuals = actuals    # the same dict for every instance


a = RegionPlan("Europe", 120_000.0)
b = RegionPlan("Asia", 90_000.0)
a.actuals["Jan"] = 9_500.0
b.actuals                         # {'Jan': 9500.0} — shared with a


# Correct
class RegionPlan:
    def __init__(self, name: str, target: float) -> None:
        self.name = name
        self.target = target
        self.actuals: dict[str, float] = {}    # fresh dict per instance

**Defining attributes outside `__init__`.** It is legal to add
attributes from anywhere, but doing so fragments the class's shape
and produces `AttributeError` on instances where the relevant code
has not run yet.

In [ ]:
# Wrong
class RegionPlan:
    def __init__(self, name: str, target: float) -> None:
        self.name = name
        self.target = target

    def record(self, month: str, value: float) -> None:
        if not hasattr(self, "actuals"):
            self.actuals = {}     # only created on first record call
        self.actuals[month] = value


plan = RegionPlan("Europe", 120_000.0)
print(plan.actuals)               # AttributeError


# Correct
class RegionPlan:
    def __init__(self, name: str, target: float) -> None:
        self.name = name
        self.target = target
        self.actuals: dict[str, float] = {}    # always present

    def record(self, month: str, value: float) -> None:
        self.actuals[month] = value

**Calling a method without parentheses.** Methods are accessed by
name and called with parentheses. Without the parentheses, the result
is the method object itself, not its return value.

In [ ]:
# Wrong
plan = RegionPlan("Europe", 120_000.0)
print(plan.total)
# <bound method RegionPlan.total of <RegionPlan ...>>


# Correct
print(plan.total())
# 0.0

The first form is sometimes useful (passing the method as a callback,
for instance), but in everyday code the missing `()` is almost always
an oversight.